# Fraud Detection Pipeline — SageMaker Studio Entry Point

Run these cells in order from SageMaker Studio. Make sure you've already:
1. Edited `src/config.py` with your bucket name and IAM role ARN
2. Downloaded `creditcard.csv` and uploaded it to `s3://<bucket>/fraud-detection/raw/creditcard.csv`

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
import sys
sys.path.append('..')

from src import config
from pipeline.pipeline import get_pipeline

session = config.get_session()
pipeline = get_pipeline(session=session, role=config.ROLE)
pipeline.upsert(role_arn=config.ROLE)
print('Pipeline created/updated:', pipeline.name)

## Start the pipeline run

This kicks off processing -> HPO tuning (10 jobs) -> evaluation -> conditional
registration. Expect this to take roughly 20-40 minutes depending on tuning
job parallelism and queue times.

In [ ]:
execution = pipeline.start()
execution.wait(delay=30, max_attempts=120)
execution.list_steps()

## Inspect the registered model

If PR-AUC cleared the threshold in `pipeline.py`, a new model package version
will show up here with status `PendingManualApproval`.

In [ ]:
import boto3
sm_client = boto3.client('sagemaker')

response = sm_client.list_model_packages(
    ModelPackageGroupName=config.MODEL_PACKAGE_GROUP_NAME,
    SortBy='CreationTime',
    SortOrder='Descending',
)
response['ModelPackageSummaryList']

## Approve the model (manual gate before deployment)

In [ ]:
model_package_arn = response['ModelPackageSummaryList'][0]['ModelPackageArn']

sm_client.update_model_package(
    ModelPackageArn=model_package_arn,
    ModelApprovalStatus='Approved',
)

## Deploy a real-time endpoint

In [ ]:
from sagemaker import ModelPackage

model = ModelPackage(
    role=config.ROLE,
    model_package_arn=model_package_arn,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=config.INFERENCE_INSTANCE_TYPE,
    endpoint_name='fraud-detection-endpoint',
)

## Score a sample transaction

Replace the values below with a real row from your test set (features only,
no label column).

In [ ]:
sample_transaction = '0.1,-1.2,0.5,...'  # V1..V28, Amount -- 29 values total

predictor.serializer = predictor.serializer  # csv serializer is default for xgboost
result = predictor.predict(sample_transaction)
print(result)

## Batch Transform (bulk scoring)

Upload a CSV of feature rows (no label) to `config.BATCH_INPUT_S3`, then run:

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type=config.BATCH_TRANSFORM_INSTANCE_TYPE,
    output_path=config.BATCH_OUTPUT_S3,
)

transformer.transform(
    data=config.BATCH_INPUT_S3,
    content_type='text/csv',
    split_type='Line',
)
transformer.wait()

## Clean up

Run this when you're done to avoid ongoing endpoint charges.

In [ ]:
predictor.delete_endpoint()